# Delta Hedging Simulation

This notebook builds the project files, runs the simulation, and produces the figures and summaries used in the report.

## Project Statement

Delta hedging attempts to neutralize price sensitivity.

Tasks:
1. Simulate stock price paths.
2. Sell an option and hedge using the delta of the option.
3. Rebalance the hedge periodically.
4. Track hedging error over time.

In [ ]:
from pathlib import Path

Path('src').mkdir(exist_ok=True)
Path('imgs').mkdir(exist_ok=True)
Path('results').mkdir(exist_ok=True)
Path('src/__init__.py').touch()


In [ ]:
%%writefile src/gbm.py
'''
src/gbm.py
Generate stock price paths and time steps for delta hedging using Geometric Brownian Motion
'''

from __future__ import annotations
from dataclasses import dataclass
from typing import Optional
import numpy as np
import matplotlib.pyplot as plt

@dataclass
class GBMConfig:
    '''Configuration for GBM simulation'''
    S0: float  # Initial stock price
    r: float  # Expected return (drift)
    sigma: float  # Volatility
    T: float  # Time horizon (in years)
    N: int  # Number of time steps
    M: int  # Number of paths to simulate
    seed: Optional[int] = 42 # Random seed for reproducibility


def generate_time_grid(T: float, N: int) -> np.ndarray:
    '''
    Generate time grid for GBM simulation
    T : Total time to maturity
    N : Number of time steps
    '''
    if T <= 0:
        raise ValueError("Time to maturity T must be positive.")
    if N <= 0:
        raise ValueError("Number of time steps N must be positive.")
    return np.linspace(0.0, T, N + 1)

def simulate_gbm_paths(
        S0: float, r: float, sigma: float, T: float, N: int, M: int, seed: Optional[int] = 42
) -> np.ndarray:
    '''
    Simulate GBM paths
    '''
    if S0 <= 0:
        raise ValueError("Initial stock price S0 must be positive.")
    if sigma < 0:
        raise ValueError("Volatility sigma cannot be negative.")
    if M <= 0:
        raise ValueError("Number of paths M must be positive.")
    if N <= 0:
        raise ValueError("Number of time steps N must be positive.")
    if T <= 0:
        raise ValueError("Time to maturity T must be positive.")

    dt = T / N
    rng = np.random.default_rng(seed)

    # standard normal shocks : (M, N) shape
    Z = rng.standard_normal((M, N))

    # log-return increments
    drift = (r - 0.5 * sigma ** 2) * dt
    diffusion = sigma * np.sqrt(dt) * Z
    log_returns = drift + diffusion

    # cumulative log returns
    log_paths = np.cumsum(log_returns, axis=1)

    # prepend zeros for initial price
    log_paths = np.concatenate((np.zeros((M, 1),dtype = float), log_paths), axis=1)

    # convert log paths to price paths
    paths = S0 * np.exp(log_paths)
    return paths

def theoretical_terminal_mean(S0: float, r: float, T: float) -> float:
    '''Calculate theoretical mean of terminal stock price'''
    return S0 * np.exp(r * T)

def theoretical_terminal_variance(S0: float, r: float, sigma: float, T: float) -> float:
    '''Calculate theoretical variance of terminal stock price'''
    return S0 ** 2 * np.exp(2 * r * T) * (np.exp(sigma ** 2 * T) - 1.0)

def validate_gbm_paths(
        paths: np.ndarray,
        S0: float,
        r: float,
        sigma: float,
        T: float) -> dict:
    '''Validate simulated GBM paths against theoretical properties '''

    if paths.ndim != 2:
        raise ValueError("Paths array must be 2-dimensional (M, N+1).")

    # Calculate empirical mean and variance of terminal stock prices
    terminal_prices = paths[:, -1]
    empirical_mean = np.mean(terminal_prices)
    empirical_variance = np.var(terminal_prices, ddof=1)  # Sample variance 

    # Calculate theoretical mean and variance
    theoretical_mean = theoretical_terminal_mean(S0, r, T)
    theoretical_variance = theoretical_terminal_variance(S0, r, sigma, T)

    # Return validation results
    return {
        "empirical_mean": empirical_mean,
        "theoretical_mean": theoretical_mean,
        "empirical_variance": empirical_variance,
        "theoretical_variance": theoretical_variance,
        "mean_error": abs(empirical_mean - theoretical_mean),
        "variance_error": abs(empirical_variance - theoretical_variance),
        "relative_mean_error": abs(empirical_mean - theoretical_mean) / theoretical_mean if theoretical_mean != 0 else float('inf'),
        "relative_variance_error": abs(empirical_variance - theoretical_variance) / theoretical_variance if theoretical_variance != 0 else float('inf')
    }

def plot_sample_paths(
        paths: np.ndarray,
        T: float,
        num_paths : int = 10,
        title: str = "Sample GBM Paths") -> None:
    '''Plot subset of simulated GBM paths'''

    if paths.ndim != 2:
        raise ValueError("Paths array must be 2-dimensional (M, N+1).")


    M, N_plus_1 = paths.shape
    if M == 0:
        raise ValueError("No paths to plot.")
    
    n_plot = min(num_paths, M)
    time_grid = generate_time_grid(T, N_plus_1 - 1) 
    plt.figure(figsize=(10, 6))
    for i in range(n_plot):
        plt.plot(time_grid, paths[i, :], linewidth=1.2)
    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel("Stock Price")
    plt.grid(True, alpha = 0.3)
    plt.tight_layout()
    plt.savefig("imgs/sample_gbm_paths.png")
    plt.show()
    

def demo()-> None:
    ''' 
    Run a simple demo 
    - simulate paths
    - validate terminal moments
    - plot sample paths
    '''

    cfg = GBMConfig(
        S0=100.0,
        r=0.05,
        sigma=0.2,
        T=1.0,
        N=252, # daily steps for 1 year 
        M=10000, # Monte Carlo paths
        seed=42
    )

    time_grid = generate_time_grid(cfg.T, cfg.N)
    paths = simulate_gbm_paths(cfg.S0, cfg.r, cfg.sigma, cfg.T, cfg.N, cfg.M, cfg.seed)
    summary = validate_gbm_paths(paths, cfg.S0, cfg.r, cfg.sigma, cfg.T)
    
    print("Time Grid:", len(time_grid), "steps from 0 to", cfg.T)
    print("Time interval dt:", time_grid[1] - time_grid[0])

    print("Validation Summary:")
    for key, value in summary.items():
        print(f"{key}: {value:.4f}")
    
    # save summary as a table 
    with open("imgs/gbm_validation_summary.txt", "w") as f:
        f.write("Validation Summary:\n")
        for key, value in summary.items():
            f.write(f"{key}: {value:.4f}\n")

    print("Plotting sample paths...")
    plot_sample_paths(paths, cfg.T, num_paths=50, title="Sample GBM Paths")

if __name__ == "__main__":
    demo()


In [ ]:
%%writefile src/bs_model.py
from __future__ import annotations

from math import erf, sqrt

import numpy as np


def normal_cdf(x: np.ndarray | float) -> np.ndarray:
    values = np.asarray(x, dtype=float)
    erf_values = np.vectorize(erf)(values / sqrt(2.0))
    return 0.5 * (1.0 + erf_values)


def d1(S: np.ndarray | float, K: float, T: np.ndarray | float, r: float, sigma: float) -> np.ndarray:
    S_values = np.asarray(S, dtype=float)
    T_values = np.maximum(np.asarray(T, dtype=float), 1e-12)
    return (np.log(S_values / K) + (r + 0.5 * sigma ** 2) * T_values) / (sigma * np.sqrt(T_values))


def d2(S: np.ndarray | float, K: float, T: np.ndarray | float, r: float, sigma: float) -> np.ndarray:
    T_values = np.maximum(np.asarray(T, dtype=float), 1e-12)
    return d1(S, K, T_values, r, sigma) - sigma * np.sqrt(T_values)


def bs_call_price(S: np.ndarray | float, K: float, T: np.ndarray | float, r: float, sigma: float) -> np.ndarray:
    S_values = np.asarray(S, dtype=float)
    T_values = np.asarray(T, dtype=float)
    intrinsic_value = np.maximum(S_values - K, 0.0)

    D1 = d1(S_values, K, T_values, r, sigma)
    D2 = d2(S_values, K, T_values, r, sigma)
    call_price = S_values * normal_cdf(D1) - K * np.exp(-r * np.maximum(T_values, 0.0)) * normal_cdf(D2)
    return np.where(T_values <= 0, intrinsic_value, call_price)


def bs_delta(S: np.ndarray | float, K: float, T: np.ndarray | float, r: float, sigma: float) -> np.ndarray:
    S_values = np.asarray(S, dtype=float)
    T_values = np.asarray(T, dtype=float)
    D1 = d1(S_values, K, T_values, r, sigma)
    delta = normal_cdf(D1)
    return np.where(T_values <= 0, np.where(S_values > K, 1.0, 0.0), delta)


def validate_black_scholes(K: float, T: float, r: float, sigma: float) -> dict:
    stock_grid = np.linspace(50.0, 150.0, 101)
    prices = bs_call_price(stock_grid, K, T, r, sigma)
    deltas = bs_delta(stock_grid, K, T, r, sigma)

    return {
        "initial_call_price": float(bs_call_price(100.0, K, T, r, sigma)),
        "initial_delta": float(bs_delta(100.0, K, T, r, sigma)),
        "min_price": float(np.min(prices)),
        "max_price": float(np.max(prices)),
        "min_delta": float(np.min(deltas)),
        "max_delta": float(np.max(deltas)),
        "price_increases_with_stock": bool(np.all(np.diff(prices) >= -1e-10)),
        "delta_between_zero_and_one": bool(np.all((deltas >= -1e-10) & (deltas <= 1.0 + 1e-10))),
    }


In [ ]:
%%writefile src/hedging.py
from __future__ import annotations

import numpy as np

from .bs_model import bs_call_price, bs_delta


def delta_hedge_single_path(S_path: np.ndarray, K: float, T: float, r: float, sigma: float, dt: float) -> dict:
    N = len(S_path) - 1
    time = np.linspace(0.0, T, N + 1)

    option_values = np.zeros(N + 1)
    deltas = np.zeros(N + 1)
    stock_positions = np.zeros(N + 1)
    cash_positions = np.zeros(N + 1)
    portfolio_values = np.zeros(N + 1)

    option_values[0] = float(bs_call_price(S_path[0], K, T, r, sigma))
    deltas[0] = float(bs_delta(S_path[0], K, T, r, sigma))
    stock_positions[0] = deltas[0]
    cash_positions[0] = option_values[0] - stock_positions[0] * S_path[0]
    portfolio_values[0] = cash_positions[0] + stock_positions[0] * S_path[0] - option_values[0]

    for t in range(1, N + 1):
        tau = max(T - t * dt, 0.0)
        cash_positions[t] = cash_positions[t - 1] * np.exp(r * dt)
        stock_positions[t] = stock_positions[t - 1]

        if t < N:
            new_delta = float(bs_delta(S_path[t], K, tau, r, sigma))
            delta_change = new_delta - stock_positions[t]
            cash_positions[t] -= delta_change * S_path[t]
            stock_positions[t] = new_delta
            deltas[t] = new_delta
        else:
            deltas[t] = stock_positions[t]

        option_values[t] = float(bs_call_price(S_path[t], K, tau, r, sigma))
        portfolio_values[t] = cash_positions[t] + stock_positions[t] * S_path[t] - option_values[t]

    payoff = max(S_path[-1] - K, 0.0)
    final_value = cash_positions[-1] + stock_positions[-1] * S_path[-1] - payoff

    return {
        "time": time,
        "option_values": option_values,
        "deltas": deltas,
        "stock_positions": stock_positions,
        "cash_positions": cash_positions,
        "portfolio_values": portfolio_values,
        "payoff": payoff,
        "final_value": final_value,
        "error": final_value,
    }


def compute_final_pnl(S_path: np.ndarray, K: float, T: float, r: float, sigma: float, dt: float) -> tuple[float, float, float]:
    hedge = delta_hedge_single_path(S_path, K, T, r, sigma, dt)
    return hedge["final_value"], hedge["payoff"], hedge["error"]


def run_delta_hedge(paths: np.ndarray, K: float, T: float, r: float, sigma: float, rebalance_every: int = 1) -> dict:
    if paths.ndim != 2:
        raise ValueError("paths must have shape (M, N + 1).")
    if rebalance_every <= 0:
        raise ValueError("rebalance_every must be positive.")

    M, N_plus_1 = paths.shape
    N = N_plus_1 - 1
    dt = T / N
    time_grid = np.linspace(0.0, T, N_plus_1)

    stock_positions = np.zeros((M, N_plus_1))
    cash_positions = np.zeros((M, N_plus_1))
    option_values = np.zeros((M, N_plus_1))
    portfolio_values = np.zeros((M, N_plus_1))
    delta_history = np.zeros((M, N_plus_1))

    option_values[:, 0] = bs_call_price(paths[:, 0], K, T, r, sigma)
    delta_history[:, 0] = bs_delta(paths[:, 0], K, T, r, sigma)
    stock_positions[:, 0] = delta_history[:, 0]
    cash_positions[:, 0] = option_values[:, 0] - stock_positions[:, 0] * paths[:, 0]
    portfolio_values[:, 0] = cash_positions[:, 0] + stock_positions[:, 0] * paths[:, 0] - option_values[:, 0]

    for t in range(1, N_plus_1):
        tau = max(T - t * dt, 0.0)
        cash_positions[:, t] = cash_positions[:, t - 1] * np.exp(r * dt)
        stock_positions[:, t] = stock_positions[:, t - 1]

        if t < N and t % rebalance_every == 0:
            new_delta = bs_delta(paths[:, t], K, tau, r, sigma)
            delta_change = new_delta - stock_positions[:, t]
            cash_positions[:, t] -= delta_change * paths[:, t]
            stock_positions[:, t] = new_delta
            delta_history[:, t] = new_delta
        else:
            delta_history[:, t] = stock_positions[:, t]

        option_values[:, t] = bs_call_price(paths[:, t], K, tau, r, sigma)
        portfolio_values[:, t] = cash_positions[:, t] + stock_positions[:, t] * paths[:, t] - option_values[:, t]

    payoff = np.maximum(paths[:, -1] - K, 0.0)
    final_values = cash_positions[:, -1] + stock_positions[:, -1] * paths[:, -1] - payoff
    errors = final_values.copy()

    return {
        "time_grid": time_grid,
        "portfolio_values": portfolio_values,
        "cash_positions": cash_positions,
        "stock_positions": stock_positions,
        "option_values": option_values,
        "delta_history": delta_history,
        "final_values": final_values,
        "errors": errors,
        "payoffs": payoff,
        "summary": {
            "mean_hedging_error": float(np.mean(errors)),
            "std_hedging_error": float(np.std(errors, ddof=1)),
            "min_hedging_error": float(np.min(errors)),
            "max_hedging_error": float(np.max(errors)),
            "rmse_hedging_error": float(np.sqrt(np.mean(errors ** 2))),
            "mean_final_pnl": float(np.mean(final_values)),
            "probability_of_loss": float(np.mean(final_values < 0.0)),
        },
    }


In [ ]:
%%writefile src/simulation.py
from __future__ import annotations

from .gbm import simulate_gbm_paths
from .hedging import run_delta_hedge


def run_simulation(
    S0: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    N: int,
    M: int,
    seed: int = 42,
    rebalance_every: int = 1,
) -> tuple:
    paths = simulate_gbm_paths(S0, r, sigma, T, N, M, seed)
    hedge_data = run_delta_hedge(paths, K, T, r, sigma, rebalance_every=rebalance_every)
    return paths, hedge_data


In [ ]:
%%writefile src/experiments.py
from __future__ import annotations

import numpy as np

from .simulation import run_simulation


def compute_stats(errors: np.ndarray) -> dict:
    return {
        "mean": float(np.mean(errors)),
        "std": float(np.std(errors, ddof=1)),
        "rmse": float(np.sqrt(np.mean(errors ** 2))),
        "mean_abs_error": float(np.mean(np.abs(errors))),
    }


def experiment_hedge_frequency(S0: float, K: float, T: float, r: float, sigma: float, M: int, seed: int = 42) -> dict:
    frequencies = {
        "Daily": 252,
        "Weekly": 52,
        "Monthly": 12
    }

    results = {}

    for label, N in frequencies.items():
        _, hedge_data = run_simulation(S0, K, T, r, sigma, N, M, seed=seed)
        results[label] = compute_stats(hedge_data["errors"])

    return results


def experiment_volatility(S0: float, K: float, T: float, r: float, N: int, M: int, seed: int = 42) -> dict:
    sigmas = {
        "Low Volatility": 0.15,
        "High Volatility": 0.35
    }

    results = {}

    for label, sigma in sigmas.items():
        _, hedge_data = run_simulation(S0, K, T, r, sigma, N, M, seed=seed)
        results[label] = compute_stats(hedge_data["errors"])

    return results


def experiment_moneyness(S0: float, T: float, r: float, sigma: float, N: int, M: int, seed: int = 42) -> dict:
    strikes = {
        "ITM": 0.9 * S0,
        "ATM": 1.0 * S0,
        "OTM": 1.1 * S0
    }

    results = {}

    for label, K in strikes.items():
        _, hedge_data = run_simulation(S0, K, T, r, sigma, N, M, seed=seed)
        results[label] = compute_stats(hedge_data["errors"])

    return results


In [ ]:
%%writefile src/plots.py
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from .bs_model import bs_delta


def plot_single_path(S_path: np.ndarray, hedge_data: dict, T: float, output_path: str) -> None:
    time = hedge_data["time"]

    fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

    axes[0].plot(time, S_path, color="navy", linewidth=2)
    axes[0].set_ylabel("Stock Price")
    axes[0].set_title("Single Path Delta Hedging")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(time, hedge_data["deltas"], color="darkorange", linewidth=2)
    axes[1].set_ylabel("Delta")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(time, hedge_data["portfolio_values"], color="darkgreen", linewidth=2)
    axes[2].axhline(0.0, color="black", linestyle="--", linewidth=1)
    axes[2].set_xlabel("Time")
    axes[2].set_ylabel("Portfolio Value")
    axes[2].grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def plot_error_histogram(errors: np.ndarray, output_path: str) -> None:
    plt.figure(figsize=(10, 6))
    plt.hist(errors, bins=50, edgecolor="black", alpha=0.8)
    plt.axvline(np.mean(errors), color="red", linestyle="--", linewidth=2)
    plt.title("Distribution of Hedging Error")
    plt.xlabel("Hedging Error")
    plt.ylabel("Frequency")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def plot_portfolio_paths(time_grid: np.ndarray, portfolio_values: np.ndarray, output_path: str, num_paths: int = 20) -> None:
    plt.figure(figsize=(10, 6))
    for i in range(min(num_paths, portfolio_values.shape[0])):
        plt.plot(time_grid, portfolio_values[i], linewidth=1.0, alpha=0.8)
    plt.axhline(0.0, color="black", linestyle="--", linewidth=1)
    plt.title("Portfolio Value Through Time")
    plt.xlabel("Time")
    plt.ylabel("Portfolio Value")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def plot_average_hedging_error(time_grid: np.ndarray, portfolio_values: np.ndarray, output_path: str) -> None:
    average_error = np.mean(portfolio_values, axis=0)

    plt.figure(figsize=(10, 6))
    plt.plot(time_grid, average_error, color="purple", linewidth=2)
    plt.axhline(0.0, color="black", linestyle="--", linewidth=1)
    plt.title("Average Hedging Error Through Time")
    plt.xlabel("Time")
    plt.ylabel("Average Hedging Error")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def plot_frequency_vs_error(results: dict, output_path: str) -> None:
    labels = list(results.keys())
    stds = [results[key]["std"] for key in labels]

    plt.figure(figsize=(8, 5))
    plt.plot(labels, stds, marker="o", linewidth=2, color="maroon")
    plt.title("Rebalancing Frequency and Hedging Error")
    plt.xlabel("Rebalancing Frequency")
    plt.ylabel("Standard Deviation of Error")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def plot_volatility_vs_error(results: dict, output_path: str) -> None:
    labels = list(results.keys())
    stds = [results[key]["std"] for key in labels]

    plt.figure(figsize=(8, 5))
    plt.bar(labels, stds, color=["#6baed6", "#fb6a4a"])
    plt.title("Volatility and Hedging Error")
    plt.xlabel("Scenario")
    plt.ylabel("Standard Deviation of Error")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def plot_moneyness_vs_error(results: dict, output_path: str) -> None:
    labels = list(results.keys())
    stds = [results[key]["std"] for key in labels]

    plt.figure(figsize=(8, 5))
    plt.bar(labels, stds, color=["#74c476", "#9e9ac8", "#fd8d3c"])
    plt.title("Moneyness and Hedging Error")
    plt.xlabel("Moneyness")
    plt.ylabel("Standard Deviation of Error")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


In [ ]:
%%writefile main.py
from __future__ import annotations

import os
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "delta-hedging-mpl"))

from src.bs_model import validate_black_scholes
from src.experiments import experiment_hedge_frequency, experiment_moneyness, experiment_volatility
from src.gbm import GBMConfig, simulate_gbm_paths, validate_gbm_paths, plot_sample_paths
from src.hedging import delta_hedge_single_path, run_delta_hedge
from src.plots import (
    plot_average_hedging_error,
    plot_error_histogram,
    plot_frequency_vs_error,
    plot_moneyness_vs_error,
    plot_portfolio_paths,
    plot_single_path,
    plot_volatility_vs_error,
)


def ensure_output_dirs() -> None:
    Path("imgs").mkdir(parents=True, exist_ok=True)
    Path("results").mkdir(parents=True, exist_ok=True)


def save_summary(summary: dict, output_path: str, title: str) -> None:
    lines = [title]
    for key, value in summary.items():
        if isinstance(value, float):
            lines.append(f"{key}: {value:.6f}")
        else:
            lines.append(f"{key}: {value}")
    Path(output_path).write_text("\n".join(lines) + "\n", encoding="utf-8")


def save_experiment_results(results: dict, output_path: str, title: str) -> None:
    lines = [title]
    for label, stats in results.items():
        lines.append("")
        lines.append(label)
        for key, value in stats.items():
            lines.append(f"{key}: {value:.6f}")
    Path(output_path).write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_report(
    gbm_summary: dict,
    bs_summary: dict,
    hedge_summary: dict,
    frequency_results: dict,
    volatility_results: dict,
    moneyness_results: dict,
) -> None:
    report_lines = [
        "# Delta Hedging Simulation",
        "",
        "## Project Statement",
        "Delta hedging attempts to neutralize price sensitivity by adjusting the stock position against an option position.",
        "",
        "This project studies four tasks:",
        "1. Simulate stock price paths.",
        "2. Sell a call option and hedge with the option delta.",
        "3. Rebalance the hedge periodically.",
        "4. Track hedging error over time.",
        "",
        "## Model Setup",
        "- Stock paths follow geometric Brownian motion.",
        "- Option prices and deltas are computed using the Black-Scholes formula.",
        "- The hedge is rebalanced at discrete times.",
        "",
        "## GBM Validation",
    ]

    for key, value in gbm_summary.items():
        report_lines.append(f"- {key}: {value:.6f}")

    report_lines.extend(["", "## Black-Scholes Checks"])
    for key, value in bs_summary.items():
        if isinstance(value, float):
            report_lines.append(f"- {key}: {value:.6f}")
        else:
            report_lines.append(f"- {key}: {value}")

    report_lines.extend(["", "## Hedging Results"])
    for key, value in hedge_summary.items():
        report_lines.append(f"- {key}: {value:.6f}")

    report_lines.extend(["", "## Rebalancing Frequency"])
    for label, stats in frequency_results.items():
        report_lines.append(
            f"- {label}: std={stats['std']:.6f}, rmse={stats['rmse']:.6f}, mean abs error={stats['mean_abs_error']:.6f}"
        )

    report_lines.extend(["", "## Volatility"])
    for label, stats in volatility_results.items():
        report_lines.append(
            f"- {label}: std={stats['std']:.6f}, rmse={stats['rmse']:.6f}, mean abs error={stats['mean_abs_error']:.6f}"
        )

    report_lines.extend(["", "## Moneyness"])
    for label, stats in moneyness_results.items():
        report_lines.append(
            f"- {label}: std={stats['std']:.6f}, rmse={stats['rmse']:.6f}, mean abs error={stats['mean_abs_error']:.6f}"
        )

    report_lines.extend(
        [
            "",
            "## Discussion",
            "- More frequent rebalancing lowers the spread of the hedging error.",
            "- Higher volatility increases the hedging error because the portfolio must react to larger price moves.",
            "- Moneyness changes the delta profile and therefore changes hedge quality.",
        ]
    )

    Path("results/project_report.md").write_text("\n".join(report_lines) + "\n", encoding="utf-8")


def print_summary(title: str, summary: dict) -> None:
    print(title)
    for key, value in summary.items():
        if isinstance(value, float):
            print(f"{key}: {value:.6f}")
        else:
            print(f"{key}: {value}")
    print()


def print_experiment_results(title: str, results: dict) -> None:
    print(title)
    for label, stats in results.items():
        print(label)
        for key, value in stats.items():
            print(f"{key}: {value:.6f}")
        print()


def main() -> None:
    ensure_output_dirs()

    cfg = GBMConfig(
        S0=100.0,
        r=0.05,
        sigma=0.2,
        T=1.0,
        N=252,
        M=10000,
        seed=42,
    )
    K = 100.0

    paths = simulate_gbm_paths(cfg.S0, cfg.r, cfg.sigma, cfg.T, cfg.N, cfg.M, cfg.seed)
    gbm_summary = validate_gbm_paths(paths, cfg.S0, cfg.r, cfg.sigma, cfg.T)
    bs_summary = validate_black_scholes(K, cfg.T, cfg.r, cfg.sigma)
    hedge_data = run_delta_hedge(paths, K, cfg.T, cfg.r, cfg.sigma)
    single_path_data = delta_hedge_single_path(paths[0], K, cfg.T, cfg.r, cfg.sigma, cfg.T / cfg.N)

    frequency_results = experiment_hedge_frequency(cfg.S0, K, cfg.T, cfg.r, cfg.sigma, cfg.M, seed=cfg.seed)
    volatility_results = experiment_volatility(cfg.S0, K, cfg.T, cfg.r, cfg.N, cfg.M, seed=cfg.seed)
    moneyness_results = experiment_moneyness(cfg.S0, cfg.T, cfg.r, cfg.sigma, cfg.N, cfg.M, seed=cfg.seed)

    save_summary(gbm_summary, "results/gbm_summary.txt", "GBM Validation")
    save_summary(bs_summary, "results/black_scholes_summary.txt", "Black-Scholes Checks")
    save_summary(hedge_data["summary"], "results/hedging_summary.txt", "Delta Hedging Summary")
    save_experiment_results(frequency_results, "results/frequency_experiment.txt", "Rebalancing Frequency")
    save_experiment_results(volatility_results, "results/volatility_experiment.txt", "Volatility Scenarios")
    save_experiment_results(moneyness_results, "results/moneyness_experiment.txt", "Moneyness Scenarios")

    plot_sample_paths(paths, cfg.T, num_paths=50, title="Simulated GBM Price Paths")
    plot_single_path(paths[0], single_path_data, cfg.T, "imgs/single_path_diagnostics.png")
    plot_error_histogram(hedge_data["errors"], "imgs/hedging_error_histogram.png")
    plot_portfolio_paths(hedge_data["time_grid"], hedge_data["portfolio_values"], "imgs/portfolio_value_paths.png")
    plot_average_hedging_error(hedge_data["time_grid"], hedge_data["portfolio_values"], "imgs/average_hedging_error.png")
    plot_frequency_vs_error(frequency_results, "imgs/frequency_vs_error.png")
    plot_volatility_vs_error(volatility_results, "imgs/volatility_vs_error.png")
    plot_moneyness_vs_error(moneyness_results, "imgs/moneyness_vs_error.png")

    write_report(
        gbm_summary,
        bs_summary,
        hedge_data["summary"],
        frequency_results,
        volatility_results,
        moneyness_results,
    )

    print_summary("GBM validation", gbm_summary)
    print_summary("Black-Scholes checks", bs_summary)
    print_summary("Delta hedging summary", hedge_data["summary"])
    print_experiment_results("Rebalancing frequency", frequency_results)
    print_experiment_results("Volatility scenarios", volatility_results)
    print_experiment_results("Moneyness scenarios", moneyness_results)


if __name__ == "__main__":
    main()


In [ ]:
!python main.py

## Text Summaries

In [ ]:
for path in [
    'results/gbm_summary.txt',
    'results/black_scholes_summary.txt',
    'results/hedging_summary.txt',
    'results/frequency_experiment.txt',
    'results/volatility_experiment.txt',
    'results/moneyness_experiment.txt',
]:
    print('\n' + '=' * 80)
    print(path)
    print('=' * 80)
    print(Path(path).read_text())


## Figures

In [ ]:
from IPython.display import Image, display

for path in [
    'imgs/sample_gbm_paths.png',
    'imgs/single_path_diagnostics.png',
    'imgs/hedging_error_histogram.png',
    'imgs/portfolio_value_paths.png',
    'imgs/average_hedging_error.png',
    'imgs/frequency_vs_error.png',
    'imgs/volatility_vs_error.png',
    'imgs/moneyness_vs_error.png',
]:
    display(Image(filename=path))
